In [53]:
import pandas as pd
import re
from IPython.display import display
from pathlib import Path
from collections import defaultdict

In [54]:
pop = pd.read_csv(r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\raw\population_addresses_messy.csv")
pop.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,student,7917 J St,Eureka,CA,95501,6863 Cedar Ln,Redding,CA,96001,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,McKinleyville,ca,95519,8942 Birch Pl,La Mesa,CA,91942,SanDiego
2,POP_S_4478952,Student,9921 Harris St,EUREKA,CA,95501,9527 Maple Dr,Redding,CA,96001,NorCal
3,POP_S_2541649,Student,3397 5th St,Eureka,CA,95501,1318 Meadow Blvd,National City,CA,91950,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,CA,95501,3406 Meadow Pl,National City,CA,91950,SanDiego


In [55]:
summary = pd.DataFrame({
    "unique_values": pop.nunique(),
    "total_rows": len(pop),
    "all_values_unique": pop.apply(lambda col: col.is_unique),
    "missing_values": pop.isna().sum()
})

print(summary)

                  unique_values  total_rows  all_values_unique  missing_values
pop_id                     7629        7785              False               0
role                         13        7785              False               0
current_street             7449        7785              False              65
current_city                 25        7785              False              66
current_state                 6        7785              False               0
current_zip                  33        7785              False              86
permanent_street           7117        7785              False              56
permanent_city               76        7785              False               0
permanent_state              10        7785              False               0
permanent_zip                38        7785              False               0
permanent_region              6        7785              False               0


In [56]:
# Data Auddit
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

In [57]:
#pop id
print("=" * 80)
print("1. DUPLICATE pop_id VALUES")
print("=" * 80)

duplicate_ids = pop[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

if duplicate_ids.empty:
    print("No duplicate pop_id values found.")
else:
    print("Repeated IDs and their occurrence counts:")
    display(
        duplicate_ids["pop_id"]
        .value_counts()
        .rename_axis("pop_id")
        .reset_index(name="occurrences")
    )

    print("\nAll rows associated with repeated IDs:")
    display(duplicate_ids)


1. DUPLICATE pop_id VALUES
Repeated IDs and their occurrence counts:


,pop_id,occurrences
0,POP_S_1371371,3
1,POP_S_4940241,3
2,POP_F_1451911,2
3,POP_F_2001167,2
4,POP_F_2930051,2
5,POP_F_4417903,2
6,POP_F_7389365,2
7,POP_F_7849184,2
8,POP_F_8236540,2
9,POP_F_8683616,2



All rows associated with repeated IDs:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
7710,POP_F_1451911,Faculty,1100 G St,Arcata,CA,95521,1100 G St,ARCATA,CA,95521,Local
6160,POP_F_1451911,Faculty,1100 G St,Arcata,CA,95521,1100 G St,ARCATA,CA,95521,Local
7654,POP_F_2001167,Faculty,7238 Greenwood Ave,Blue Lake,CA,95525,7238 Greenwood Ave,Blue Lake,CA,95525,Local
6305,POP_F_2001167,Faculty,7238 Greenwood Ave,Blue Lake,CA,95525,7238 Greenwood Ave,Blue Lake,CA,95525,Local
7684,POP_F_2930051,Faculty,366 Q St,Arcata,CA,95521,366 Q St,Arcata,CA,95521,Local
...,...,...,...,...,...,...,...,...,...,...,...
3135,POP_S_9948521,Student,6124 Central Ave,McKinleyville,CA,95519,8447 Vista St,Phoenix,AZ,85004,OutOfState
1896,POP_S_9952831,Student,9825 Union St,Arcata,CA,95521,7419 Pine St,San Jose,CA,95112,BayArea
7664,POP_S_9952831,Student,9825 Union St,Arcata,CA,95521,7419 Pine St,San Jose,CA,95112,BayArea
7706,POP_S_9973271,Student,5488 h st,Arcata,CA,95521,5485 Canyon Dr,Chico,CA,95926,NorCal


In [58]:
#drop duplicate rows
pop_unique = pop.drop_duplicates()
#non-duplicate rows where pop_id is not unique (how do we handle?)
pop_unique[pop_unique.duplicated(subset = "pop_id", keep = False)].sort_values("pop_id")

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1427,POP_S_1371371,Student,5576 G St,Arcata,CA,95521,282 Hill Ln,Glendale,CA,91203,LA
7163,POP_S_1371371,Staff/Admin,2570 Myrtle Ave,Eureka,CA,95501,2570 Myrtle Ave,EUREKA,CA,95501,Local
380,POP_S_4940241,Student,8640 Janis Ct,Arcata,OR,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,STUDENT,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
3880,POP_S_6159896,Student,1695 G St,Arcata,CA,95521,2144 Willow Way,Austin,TX,78701,OutOfState
7097,POP_S_6159896,Staff/Admin,8317 WINCHESTER AVE,McKinleyville,CA,95519,5317 Winchester Ave,McKinleyville,CA,95519,Local
43,POP_S_7366358,Student,2208 Baldwin St,NaN,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,Student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal


In [59]:
#check that all remaining pop_id are of the correct format
pop_unique[pop_unique["pop_id"].astype(str).str.fullmatch(r"POP_[SFA]_\d+") == False]


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region


In [60]:
# Role audit
print("\n" + "=" * 80)
print("2. ROLE AUDIT")
print("=" * 80)

print("All role values and their frequencies:")
role_counts = (
    pop["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)
display(role_counts)

# Normalize capitalization and surrounding whitespace
role_normalized = pop["role"].astype("string").str.strip().str.casefold()

print("\nRoles after ignoring capitalization and outer whitespace:")
display(
    pop.assign(normalized_role=role_normalized)
       .groupby("normalized_role", dropna=False)
       .agg(
           count=("role", "size"),
           original_values=("role",
                            lambda x: sorted(x.dropna().unique().tolist()))
       )
       .reset_index()
)



2. ROLE AUDIT
All role values and their frequencies:


,role,count
0,Student,6164
1,Staff/Admin,683
2,Faculty,498
3,STUDENT,137
4,student,115
5,Studnet,112
6,Staff,15
7,faculty,11
8,Faculty,10
9,FACULTY,10



Roles after ignoring capitalization and outer whitespace:


,normalized_role,count,original_values
0,faculty,529,"[FACULTY, Faculty, Faculty , faculty]"
1,staff,15,[Staff]
2,staff-admin,10,[Staff-Admin]
3,staff/admin,703,"[STAFF/ADMIN, Staff/Admin, staff/admin]"
4,student,6416,"[STUDENT, Student, student]"
5,studnet,112,[Studnet]


In [61]:
#standardize role names and confirm 
pop_unique.loc[pop_unique["role"].str.startswith(("stu", "STU", "Stu")), "role"] = "student"
pop_unique.loc[pop_unique["role"].str.startswith(("sta", "STA", "Sta")), "role"] = "staff/admin"
pop_unique.loc[pop_unique["role"].str.startswith(("fac", "FAC", "Fac")), "role"] = "faculty"
pop = pop_unique
pop.query("role not in ['student', 'staff/admin', 'faculty']")


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region


In [62]:
print("All role values and their frequencies:")
role_counts = (
    pop["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)
display(role_counts)

All role values and their frequencies:


,role,count
0,student,6400
1,staff/admin,713
2,faculty,520


In [63]:
# Current Address Audit
current_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip"
]

print("\n" + "=" * 80)
print("3. CURRENT ADDRESS AUDIT")
print("=" * 80)



3. CURRENT ADDRESS AUDIT


In [64]:
# Current Address Audit
permanent_columns = [
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

print("\n" + "=" * 80)
print("4. PERMANENT ADDRESS AUDIT")
print("=" * 80)



4. PERMANENT ADDRESS AUDIT


In [65]:
def audit_duplicates(df, columns, address_type):
    for column in columns:
        print("\n" + "-" * 70)
        print(f"{address_type}: {column}")
        print("-" * 70)

        # Ignore missing values when identifying repeats
        counts = df[column].value_counts(dropna=True)
        repeated_values = counts[counts > 1]

        if repeated_values.empty:
            print("No repeated non-missing values found.")
            continue

        print("Repeated values and occurrence counts:")
        display(
            repeated_values
            .rename_axis(column)
            .reset_index(name="occurrences")
        )

        # Show every full row associated with a repeated value
        mask = (
            df[column].notna()
            & df[column].isin(repeated_values.index)
        )

        print("All rows containing those repeated values:")
        display(df.loc[mask].sort_values(column))


# Run the current address checks
audit_duplicates(pop, current_columns, "CURRENT ADDRESS")

# Run the permanent address checks
audit_duplicates(pop, permanent_columns, "PERMANENT ADDRESS")


----------------------------------------------------------------------
CURRENT ADDRESS: current_street
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_street,occurrences
0,-,20
1,unknown,18
2,8571 Q St,3
3,7550 Samoa Blvd,2
4,173 Baldwin St,2
5,8223 K St,2
6,8332 LK Wood Blvd,2
7,3454 Q St,2
8,1457 Samoa Blvd,2
9,2395 Samoa Blvd,2


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
7494,POP_S_14842217,student,-,Arcata,CA,95521,6671 Elm Ave,Escondido,CA,92025,SanDiego
5616,POP_S_6680999,student,-,Arcata,CA,95521,3291 Spruce Pl,Seattle,WA,98101,OutOfState
1377,POP_S_4268441,student,-,Eureka,CA,95501,5725 Ridge Ave,Boulder,CO,80302,OutOfState
5393,POP_S_7770946,student,-,Arcata,CA,95521,3622 Canyon Ave,Ukiah,CA,95482,NorCal
1612,POP_S_4312891,student,-,Eureka,CA,95501,2316 Lake Blvd,San Francisco,CA,94110,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
5139,POP_S_5937004,student,unknown,Eureka,CA,95501,685 Vista Blvd,Anaheim,CA,92801,LA
2374,POP_S_9046076,student,unknown,Arcata,CA,95521,5754 Pine Blvd,Seattle,WA,98101,OutOfState
4402,POP_S_2284475,student,unknown,Arcata,CA,95521,7381 Elm Rd,San Francisco,CA,94110,BayArea
536,POP_S_4571162,student,unknown,Eureka,CA,95501,2110 Maple Way,San Jose,CA,95112,BayArea



----------------------------------------------------------------------
CURRENT ADDRESS: current_city
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_city,occurrences
0,Arcata,3705
1,Eureka,1609
2,McKinleyville,1223
3,Blue Lake,215
4,Trinidad,172
5,Fortuna,166
6,arcata,66
7,ARCATRA,63
8,ARCATA,58
9,Arcatra,51


... (26 outputs left out)


----------------------------------------------------------------------
PERMANENT ADDRESS: permanent_region
----------------------------------------------------------------------
Repeated values and occurrence counts:


,permanent_region,occurrences
0,LA,1602
1,BayArea,1464
2,OutOfState,1425
3,Local,1233
4,NorCal,1124
5,SanDiego,785


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
2798,POP_S_9591254,student,666 Wabash Ave,Eureka,CA,95501,-,Daly City,CA,94015,BayArea
3873,POP_S_1855771,student,1412 I St,Eureka,CA,95501,8792 Ridge Ave,Daly City,CA,94015,BayArea
1413,POP_S_7452240,student,5253 6th St,Eureka,CA,95501,8250 Sunset Blvd,Fremont,CA,94536,BayArea
3888,POP_S_3205610,student,2979 patrick point dr,Trinidad,CA,95570,3089 Maple St,Berkeley,CA,94704,BayArea
3890,POP_S_5029427,student,3626 Janis Ct,Arcata,CA,95521,1600 Ridge Pl,Fremont,CA,94536,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
1996,POP_S_2606285,student,2985 Janis Ct,Arcata,CA,95521,8695 Maple Blvd,Chula Vista,CA,91910,SanDiego
4504,POP_S_9619821,student,4306 Union St,Arcata,CA,95521,1552 Meadow Rd,La Mesa,CA,91942,SanDiego
1987,POP_S_7056598,student,6305 K St,Eureka,CA,95501,NaN,Oceanside,CA,92054,SanDiego
4480,POP_S_1900638,student,1474 14th St,Arcata,CA,95521,4048 Elm Rd,La Mesa,CA,91942,SanDiego


In [66]:
# Find Case and Whitespace
def audit_format_variations(df, columns, address_type):
    for column in columns:
        # Convert to strings, trim outer spaces, ignore case
        normalized = (
            df[column]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        # Preserve original spellings for comparison
        temp = pd.DataFrame({
            "normalized_value": normalized,
            "original_value": df[column]
        }).dropna(subset=["normalized_value"])

        variations = (
            temp.groupby("normalized_value")["original_value"]
            .agg(lambda x: sorted(x.unique().tolist()))
        )

        # Keep only groups with multiple original representations
        variations = variations[variations.map(len) > 1]

        if not variations.empty:
            print(
                f"\n{address_type} FORMAT VARIATIONS: {column}"
            )
            display(variations.rename("original_variations").reset_index())


print("\n" + "=" * 80)
print("5. CASE AND WHITESPACE VARIATIONS")
print("=" * 80)

audit_format_variations(
    pop,
    ["role"] + current_columns + permanent_columns,
    "ROLE / ADDRESS"
)

print("\n" + "=" * 80)
print("DATA-QUALITY AUDIT COMPLETE")
print("=" * 80)


5. CASE AND WHITESPACE VARIATIONS

ROLE / ADDRESS FORMAT VARIATIONS: current_street


,normalized_value,original_variations
0,3200 janis ct,"[3200 Janis Ct, 3200 janis ct]"
1,322 h st,"[322 H St, 322 h st]"
2,3334 murray rd,"[3334 MURRAY RD, 3334 Murray Rd]"
3,4742 alliance rd,"[4742 ALLIANCE RD, 4742 Alliance Rd]"
4,5073 central ave,"[5073 Central Ave, 5073 central ave]"
5,6141 11th st,"[6141 11th St, 6141 11th st]"
6,6325 myrtle ave,"[6325 Myrtle Ave, 6325 myrtle ave]"
7,636 g st,"[636 G St, 636 g st]"
8,6453 wabash ave,"[6453 Wabash Ave, 6453 wabash ave]"
9,659 h st,"[659 H St, 659 h st]"



ROLE / ADDRESS FORMAT VARIATIONS: current_city


,normalized_value,original_variations
0,arcata,"[ARCATA, Arcata, arcata]"
1,arcatra,"[ARCATRA, Arcatra]"
2,blue lake,"[BLUE LAKE, Blue Lake]"
3,eureka,"[EUREKA, Eureka, eureka]"
4,fortuna,"[Fortuna, fortuna]"
5,mckinleyville,"[MCKINLEYVILLE, McKinleyville, mckinleyville]"
6,trinidad,"[TRINIDAD, Trinidad]"



ROLE / ADDRESS FORMAT VARIATIONS: current_state


,normalized_value,original_variations
0,ca,"[CA, Ca, ca]"



ROLE / ADDRESS FORMAT VARIATIONS: permanent_city


,normalized_value,original_variations
0,anaheim,"[ANAHEIM, Anaheim]"
1,arcata,"[ARCATA, Arcata]"
2,austin,"[AUSTIN, Austin]"
3,berkeley,"[BERKELEY, Berkeley]"
4,blue lake,"[BLUE LAKE, Blue Lake]"
5,boise,"[BOISE, Boise]"
6,boulder,"[BOULDER, Boulder]"
7,brooklyn,"[BROOKLYN, Brooklyn]"
8,chico,"[CHICO, Chico]"
9,chula vista,"[CHULA VISTA, Chula Vista]"



DATA-QUALITY AUDIT COMPLETE


# Clean address

In [67]:
print("=" * 80)
print("ADDRESS DATA CLEANING")
print("=" * 80)

# Define address columns
current_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip"
]

permanent_columns = [
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_columns = current_columns + permanent_columns


# just in case copy data to preserve previous role work.

# Create a snapshot of the current working DataFrame.
# This includes the role-cleaning changes made previously.

pop_before_address_cleaning = pop.copy(deep=True)

print("Records before address cleaning:", len(pop))
print("Address columns being processed:", len(address_columns))


# whitespace cleanup
# Remove leading and trailing whitespace from address fields.
# Preserve missing values rather than converting them to strings.

for column in address_columns:

    pop[column] = (
        pop[column]
        .astype("string")
        .str.strip()
    )


# standardize city name
# Explicit corrections for known variations.
# Keys must be lowercase, stripped representations.

city_mapping = {

    # Arcata
    "arcata": "Arcata",
    "arcatra": "Arcata",

    # Eureka
    "eureka": "Eureka",
    "eureaka": "Eureka",
    "eurika": "Eureka",

    # McKinleyville
    "mckinleyville": "McKinleyville",
    "mckinelyville": "McKinleyville",

    # Blue Lake
    "blue lake": "Blue Lake",
    "bluelake": "Blue Lake",

    # Trinidad
    "trinidad": "Trinidad",
    "trinadad": "Trinidad",

    # Fortuna
    "fortuna": "Fortuna",
    "fortunna": "Fortuna",

    # Other cities observed in the permanent address audit
    "glendale": "Glendale",
    "anaheim": "Anaheim",
    "fremont": "Fremont",
    "los angeles": "Los Angeles",
    "pasadena": "Pasadena",
    "santa monica": "Santa Monica",
    "daly city": "Daly City",
    "long beach": "Long Beach",
    "berkeley": "Berkeley",
    "san francisco": "San Francisco",
    "oakland": "Oakland",
    "san jose": "San Jose",
    "redding": "Redding",
    "ukiah": "Ukiah",
    "sacramento": "Sacramento",
    "santa rosa": "Santa Rosa",
    "chico": "Chico",
    "boulder": "Boulder",
    "salt lake city": "Salt Lake City",
    "portland": "Portland",
    "seattle": "Seattle",
    "austin": "Austin",
    "national city": "National City",
    "brooklyn": "Brooklyn",
    "phoenix": "Phoenix",
    "reno": "Reno",
    "boise": "Boise",
    "la mesa": "La Mesa",
    "oceanside": "Oceanside",
    "escondido": "Escondido",
    "chula vista": "Chula Vista",
    "san diego": "San Diego"
}


#Apply standardization
city_columns = [
    "current_city",
    "permanent_city"
]

for column in city_columns:

    normalized_city = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    # Convert known missing-value placeholders to actual missing values.
    normalized_city = normalized_city.mask(
        normalized_city.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Correct known city variations.
    # Unrecognized city names are retained for manual review.
    pop[column] = normalized_city.map(city_mapping).where(
        normalized_city.isin(city_mapping),
        pop[column]
    )

    # Standardize recognized missing-value placeholders.
    pop[column] = pop[column].mask(
        normalized_city.isin(["", "-", "unknown", "n/a", "na"])
    )


# Standard State
state_columns = [
    "current_state",
    "permanent_state"
]

state_mapping = {
    "ca": "CA",
    "california": "CA",
    "or": "OR",
    "oregon": "OR"
}

for column in state_columns:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\.$", "", regex=True)
    )

    normalized_state = normalized_state.mask(
        normalized_state.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Standardize recognized state names and abbreviations.
    # Preserve other nonmissing state values for review.
    pop[column] = normalized_state.map(state_mapping).where(
        normalized_state.isin(state_mapping),
        pop[column]
    )

    pop[column] = pop[column].mask(
        normalized_state.isin(["", "-", "unknown", "n/a", "na"])
    )

# update current_state based on current_city

# Cities that should have California as their current state.
california_cities = [
    "Arcata",
    "Eureka",
    "McKinleyville",
    "Blue Lake",
    "Trinidad",
    "Fortuna"
]

# Normalize city values for case-insensitive matching.
normalized_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Create a case-insensitive list of target cities.
target_cities = [city.casefold() for city in california_cities]

# Identify matching records.
mask = normalized_city.isin(target_cities)

# Count records that will be updated.
records_to_update = mask.sum()

# Update current_state to CA.
pop.loc[mask, "current_state"] = "CA"

print(f"Records updated: {records_to_update:,}")
print("Current state successfully standardized to CA.")

# standard street format
street_columns = [
    "current_street",
    "permanent_street"
]

for column in street_columns:

    pop[column] = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Convert blank strings and explicit placeholders to missing.
    normalized_street = pop[column].str.casefold()

    pop[column] = pop[column].mask(
        normalized_street.isin(["", "-", "unknown", "n/a", "na"])
    )

# standardize city and zip

# Define the city-to-ZIP mapping.
city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Create the reverse ZIP-to-city mapping.
zip_city_mapping = {
    zip_code: city
    for city, zip_code in city_zip_mapping.items()
}

# normalize city and zip

normalized_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

normalized_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

# update current_zip based on current_city

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

city_mask = normalized_city.isin(city_lookup)

pop.loc[city_mask, "current_zip"] = (
    normalized_city.loc[city_mask].map(city_lookup)
)

# update curent_city based on current_zip

zip_mask = normalized_zip.isin(zip_city_mapping)

pop.loc[zip_mask, "current_city"] = (
    normalized_zip.loc[zip_mask].map(zip_city_mapping)
)

# display

print("=" * 70)
print("CURRENT CITY AND ZIP STANDARDIZATION")
print("=" * 70)

print(f"Records with city-based ZIP updates: {city_mask.sum():,}")
print(f"Records with ZIP-based city updates: {zip_mask.sum():,}")

print("\nUpdated city and ZIP combinations:")

verification = (
    pop.loc[
        pop["current_city"].isin(city_zip_mapping.keys())
        | pop["current_zip"].isin(zip_city_mapping.keys()),
        ["current_city", "current_zip"]
    ]
    .value_counts()
    .reset_index(name="records")
    .sort_values(["current_city", "current_zip"])
)

display(verification)

#standard zip
zip_columns = [
    "current_zip",
    "permanent_zip"
]

for column in zip_columns:

    # Keep ZIP codes as strings to preserve leading zeros.
    zip_values = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    normalized_zip = zip_values.str.casefold()

    # Convert missing-value placeholders to actual missing values.
    zip_values = zip_values.mask(
        normalized_zip.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Remove a trailing .0 only when the entire value is numeric
    # followed by a decimal point and zero.
    zip_values = zip_values.str.replace(
        r"^(\d+)\.0$",
        r"\1",
        regex=True
    )

    # Pad numeric ZIP codes of one to four digits with leading zeros.
    # Preserve five-digit ZIP codes and ZIP+4 values.
    valid_short_zip = zip_values.str.fullmatch(r"\d{1,4}", na=False)

    zip_values.loc[valid_short_zip] = (
        zip_values.loc[valid_short_zip].str.zfill(5)
    )

    pop[column] = zip_values


# result output
print("\n" + "=" * 80)
print("ADDRESS CLEANING SUMMARY")
print("=" * 80)

print("Records after address cleaning:", len(pop))

print("\nMissing values by address column:")

display(
    pop[address_columns]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

print("\nUnique values by address column:")

display(
    pop[address_columns]
    .nunique()
    .rename("unique_values")
    .to_frame()
)

print("\nAddress cleaning completed.")
print("The cleaned values currently exist in the working DataFrame.")
print("The CSV has not yet been overwritten or exported.")

ADDRESS DATA CLEANING
Records before address cleaning: 7633
Address columns being processed: 9
Records updated: 7,527
Current state successfully standardized to CA.
CURRENT CITY AND ZIP STANDARDIZATION
Records with city-based ZIP updates: 7,527
Records with ZIP-based city updates: 7,212

Updated city and ZIP combinations:


,current_city,current_zip,records
0,Arcata,95521,3993
3,Blue Lake,95525,231
1,Eureka,95501,1744
5,Fortuna,95540,178
2,McKinleyville,95519,1303
4,Trinidad,95570,184



ADDRESS CLEANING SUMMARY
Records after address cleaning: 7633

Missing values by address column:


,missing_values
current_street,101
current_city,0
current_state,0
current_zip,0
permanent_street,111
permanent_city,0
permanent_state,0
permanent_zip,0
permanent_region,0



Unique values by address column:


,unique_values
current_street,7447
current_city,6
current_state,1
current_zip,6
permanent_street,7115
permanent_city,38
permanent_state,10
permanent_zip,38
permanent_region,6



Address cleaning completed.
The cleaned values currently exist in the working DataFrame.
The CSV has not yet been overwritten or exported.


In [68]:
# re audit
print("=" * 80)
print("CURRENT ADDRESS AUDIT AFTER CLEANING")
print("=" * 80)

audit_duplicates(
    pop,
    current_columns,
    "CURRENT ADDRESS AFTER CLEANING"
)

print("\n" + "=" * 80)
print("PERMANENT ADDRESS AUDIT AFTER CLEANING")
print("=" * 80)

audit_duplicates(
    pop,
    permanent_columns,
    "PERMANENT ADDRESS AFTER CLEANING"
)

CURRENT ADDRESS AUDIT AFTER CLEANING

----------------------------------------------------------------------
CURRENT ADDRESS AFTER CLEANING: current_street
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_street,occurrences
0,8571 Q St,3
1,7550 Samoa Blvd,2
2,173 Baldwin St,2
3,8223 K St,2
4,8332 LK Wood Blvd,2
5,3454 Q St,2
6,1457 Samoa Blvd,2
7,2395 Samoa Blvd,2
8,1021 5th St,2
9,3228 Hiller Rd,2


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1489,POP_S_6911832,student,1009 Main St,Fortuna,CA,95540,3329 Willow Blvd,Chico,CA,95926,NorCal
3705,POP_S_1072070,student,1009 Main St,Trinidad,CA,95570,9329 Birch Blvd,San Diego,CA,92103,SanDiego
4917,POP_S_7729000,student,1021 5th St,Eureka,CA,95501,7113 Hill St,Boulder,CO,80302,OutOfState
338,POP_S_7547868,student,1021 5th St,Eureka,CA,95501,1906 Ridge Ln,National City,CA,91950,SanDiego
874,POP_S_5764958,student,1087 Hiller Rd,McKinleyville,CA,95519,3376 Oak Way,Redding,CA,96001,NorCal
1752,POP_S_2084657,student,1087 Hiller Rd,McKinleyville,CA,95519,6146 Oak Rd,Oakland,CA,94612,BayArea
5584,POP_S_3378679,student,1120 11th St,Arcata,CA,95521,9919 Ridge Way,Santa Monica,CA,90401,LA
5624,POP_S_9217676,student,1120 11th St,Arcata,CA,95521,1256 Willow Ln,Redding,CA,96001,NorCal
6242,POP_F_6104810,faculty,1286 Baldwin St,Arcata,CA,95521,1286 Baldwin St,Arcata,CA,95521,Local
1757,POP_S_6143981,student,1286 Baldwin St,Arcata,CA,95521,608 Sunset St,Brooklyn,NY,11211,OutOfState



----------------------------------------------------------------------
CURRENT ADDRESS AFTER CLEANING: current_city
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_city,occurrences
0,Arcata,3993
1,Eureka,1744
2,McKinleyville,1303
3,Blue Lake,231
4,Trinidad,184
5,Fortuna,178


... (26 outputs left out)


----------------------------------------------------------------------
PERMANENT ADDRESS AFTER CLEANING: permanent_region
----------------------------------------------------------------------
Repeated values and occurrence counts:


,permanent_region,occurrences
0,LA,1602
1,BayArea,1464
2,OutOfState,1425
3,Local,1233
4,NorCal,1124
5,SanDiego,785


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
2798,POP_S_9591254,student,666 Wabash Ave,Eureka,CA,95501,<NA>,Daly City,CA,94015,BayArea
3873,POP_S_1855771,student,1412 I St,Eureka,CA,95501,8792 Ridge Ave,Daly City,CA,94015,BayArea
1413,POP_S_7452240,student,5253 6th St,Eureka,CA,95501,8250 Sunset Blvd,Fremont,CA,94536,BayArea
3888,POP_S_3205610,student,2979 patrick point dr,Trinidad,CA,95570,3089 Maple St,Berkeley,CA,94704,BayArea
3890,POP_S_5029427,student,3626 Janis Ct,Arcata,CA,95521,1600 Ridge Pl,Fremont,CA,94536,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
1996,POP_S_2606285,student,2985 Janis Ct,Arcata,CA,95521,8695 Maple Blvd,Chula Vista,CA,91910,SanDiego
4504,POP_S_9619821,student,4306 Union St,Arcata,CA,95521,1552 Meadow Rd,La Mesa,CA,91942,SanDiego
1987,POP_S_7056598,student,6305 K St,Eureka,CA,95501,<NA>,Oceanside,CA,92054,SanDiego
4480,POP_S_1900638,student,1474 14th St,Arcata,CA,95521,4048 Elm Rd,La Mesa,CA,91942,SanDiego


In [69]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 8
Unique duplicated identifiers: 4
FAIL: Duplicate identifiers remain.


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1427,POP_S_1371371,student,5576 G St,Arcata,CA,95521,282 Hill Ln,Glendale,CA,91203,LA
7163,POP_S_1371371,staff/admin,2570 Myrtle Ave,Eureka,CA,95501,2570 Myrtle Ave,Eureka,CA,95501,Local
380,POP_S_4940241,student,8640 Janis Ct,Arcata,CA,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,student,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
3880,POP_S_6159896,student,1695 G St,Arcata,CA,95521,2144 Willow Way,Austin,TX,78701,OutOfState
7097,POP_S_6159896,staff/admin,8317 WINCHESTER AVE,McKinleyville,CA,95519,5317 Winchester Ave,McKinleyville,CA,95519,Local
43,POP_S_7366358,student,2208 Baldwin St,Arcata,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal



4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


FAIL: Incorrect ID prefixes found.


,pop_id,role,expected_prefix
0,POP_S_1717516,staff/admin,POP_SA
1,POP_S_1418572,staff/admin,POP_SA
2,POP_S_3328417,staff/admin,POP_SA
3,POP_S_6650936,staff/admin,POP_SA
4,POP_S_1657976,staff/admin,POP_SA
...,...,...,...
708,POP_S_4593087,staff/admin,POP_SA
709,POP_S_6591844,staff/admin,POP_SA
710,POP_S_2353753,staff/admin,POP_SA
711,POP_S_4832818,staff/admin,POP_SA



5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
FAIL: Unique pop_id values
PASS: Recognized roles
FAIL: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET IS NOT READY FOR FINAL EXPORT.
Resolve the failed validation checks before proceeding.


In [70]:
# define prefix per role

role_prefix = {
    "student": "POP_S",
    "faculty": "POP_F",
    "staff/admin": "POP_SA"
}

# Standardize pop_id as a string and remove surrounding spaces.
pop["pop_id"] = pop["pop_id"].astype("string").str.strip()

# ID incorrect prefix

positions_to_reassign = set()

for position in range(len(pop)):

    identifier = pop.iloc[position]["pop_id"]
    role = pop.iloc[position]["role"]
    expected_prefix = role_prefix[role]

    # Require the correct prefix and a numeric suffix.
    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, identifier):
        positions_to_reassign.add(position)

# generate unique replacement ID

# Track all existing identifiers to prevent collisions.
used_ids = set(pop["pop_id"].tolist())

# Find the largest existing numeric suffix for each prefix.
max_suffix = defaultdict(int)

for identifier in used_ids:

    match = re.fullmatch(r"^(POP_[A-Z]+)_(\d+)$", identifier)

    if match:
        prefix = match.group(1)
        suffix = int(match.group(2))
        max_suffix[prefix] = max(max_suffix[prefix], suffix)

# Allocate new IDs only to the records that need correction.
pop_id_column_position = pop.columns.get_loc("pop_id")

for position in sorted(positions_to_reassign):

    role = pop.iloc[position]["role"]
    prefix = role_prefix[role]

    # Start after the largest existing suffix for this prefix.
    candidate_number = max_suffix[prefix] + 1
    new_identifier = f"{prefix}_{candidate_number}"

    # Guarantee uniqueness.
    while new_identifier in used_ids:
        candidate_number += 1
        new_identifier = f"{prefix}_{candidate_number}"

    # Assign the replacement ID.
    pop.iat[position, pop_id_column_position] = new_identifier

    # Update tracking variables.
    used_ids.add(new_identifier)
    max_suffix[prefix] = candidate_number

In [71]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 4
Unique duplicated identifiers: 2
FAIL: Duplicate identifiers remain.


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
380,POP_S_4940241,student,8640 Janis Ct,Arcata,CA,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,student,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
43,POP_S_7366358,student,2208 Baldwin St,Arcata,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal



4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


PASS: All IDs match their respective role prefixes.

5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
FAIL: Unique pop_id values
PASS: Recognized roles
PASS: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET IS NOT READY FOR FINAL EXPORT.
Resolve the failed validation checks before proceeding.


In [72]:
# assign new pop_id to duplicates.

# Preserve a backup before modifying identifiers.
pop_before_reassignment = pop.copy(deep=True)

# Ensure pop_id is stored as a string.
pop["pop_id"] = pop["pop_id"].astype("string").str.strip()

# Identify existing IDs and their numeric suffixes.
used_ids = set(pop["pop_id"].dropna())

student_ids = [
    int(match.group(1))
    for identifier in used_ids
    if (match := re.fullmatch(r"POP_S_(\d+)", identifier))
]

# Start after the largest existing student ID.
next_id = max(student_ids, default=0) + 1

# Identify duplicate IDs.
duplicate_mask = pop["pop_id"].duplicated(keep="first")

# Store the original identifiers for comparison.
original_ids = pop["pop_id"].copy()

# Assign new IDs to subsequent duplicate records.
for index in pop.index[duplicate_mask]:

    new_id = f"POP_S_{next_id}"

    # Ensure the new ID does not already exist.
    while new_id in used_ids:
        next_id += 1
        new_id = f"POP_S_{next_id}"

    # Assign the new identifier.
    pop.at[index, "pop_id"] = new_id

    # Track the new identifier.
    used_ids.add(new_id)
    next_id += 1

# show outputs

changed_records = pop.loc[
    original_ids != pop["pop_id"],
    ["pop_id", "role", "current_street",
     "current_city", "permanent_street", "permanent_city"]
].copy()

changed_records.insert(
    0,
    "original_pop_id",
    original_ids.loc[original_ids != pop["pop_id"]]
)

print("=" * 70)
print("POP_ID REASSIGNMENT RESULTS")
print("=" * 70)

print(f"New identifiers assigned: {len(changed_records)}")

display(changed_records)

# verify if unique

print("\nFINAL POP_ID VALIDATION")
print("-" * 70)

print(f"Total records: {len(pop):,}")
print(f"Missing IDs: {pop['pop_id'].isna().sum():,}")
print(f"Duplicate IDs: {pop['pop_id'].duplicated().sum():,}")

assert pop["pop_id"].notna().all(), "Missing IDs remain."
assert pop["pop_id"].is_unique, "Duplicate IDs remain."

print("\nPASS: All pop_id values are unique.")

POP_ID REASSIGNMENT RESULTS
New identifiers assigned: 2


,original_pop_id,pop_id,role,current_street,current_city,permanent_street,permanent_city
3554,POP_S_7366358,POP_S_99263434,student,2551 Union St,Arcata,421 Lake Rd,Santa Rosa
4552,POP_S_4940241,POP_S_99263435,student,1466 Spears Rd,Arcata,6155 Spruce Dr,San Diego



FINAL POP_ID VALIDATION
----------------------------------------------------------------------
Total records: 7,633
Missing IDs: 0
Duplicate IDs: 0

PASS: All pop_id values are unique.


In [73]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 0
Unique duplicated identifiers: 0
PASS: All pop_id values are unique.

4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


PASS: All IDs match their respective role prefixes.

5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
PASS: Unique pop_id values
PASS: Recognized roles
PASS: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.
NOTE: Missing values remain. Review the completeness report and determine whether they are acceptable.


# Save the New Dataset

In [74]:
# Define the output directory
output_dir = Path(
    r"C:\Users\MARE\Documents\data311_project2\data\processed"
)

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output filename
output_file = output_dir / "population_addresses_cleaned.csv"

# Save the cleaned DataFrame
pop.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("File location:", output_file)
print("Number of records saved:", len(pop))

Cleaned dataset saved successfully.
File location: C:\Users\MARE\Documents\data311_project2\data\processed\population_addresses_cleaned.csv
Number of records saved: 7633
